#Untitled2

## Static config

### Import

In [4]:
import csv
import requests
from requests.adapters import HTTPAdapter, Retry

### URL and retries config

In [5]:
retries = Retry(
    total=5,
    backoff_factor=0.25,
    status_forcelist=[500, 502, 503, 504],
)

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retries))

uniprot_url = "https://rest.uniprot.org/uniprotkb/search"

### Queries

In [6]:
positive_query = """
(reviewed:true) AND
(fragment:false) AND
(taxonomy_id:2759) AND
(existence:1) AND
(length:[40 TO *]) AND
(ft_signal_exp:*)
""".replace("\n", " ")


negative_query = """
(reviewed:true) AND
(fragment:false) AND
(taxonomy_id:2759) AND
(existence:1) AND
(length:[40 TO *]) NOT
(ft_signal:*) AND
(
    cc_scl_term_exp:SL-0091 OR
    cc_scl_term_exp:SL-0173 OR
    cc_scl_term_exp:SL-0191 OR
    cc_scl_term_exp:SL-0209 OR
    cc_scl_term_exp:SL-0204 OR
    cc_scl_term_exp:SL-0039
)
""".replace("\n", " ")

## Functions

### Yield UniProt entries across all result pages

In [7]:
def iter_uniprot_entries(query, batch_size=500):


    params = {
        "query": query,
        "format": "json",
        "compressed": "false",
        "size": batch_size,
    }

    url = uniprot_url

    while url:
        response = session.get(
            url,
            params=params if url == uniprot_url else None,
            timeout=60,
        )
        response.raise_for_status()

        yield from response.json()["results"]

        url = response.links.get("next", {}).get("url")
        params = None

### Record functions


Fields shared by both datasets.

In [8]:
def get_kingdom(entry):
    lineage = entry.get("organism", {}).get("lineage", [])

    for kingdom in ("Metazoa", "Fungi", "Viridiplantae"):
        if kingdom in lineage:
            return kingdom

    return "Other"


def base_record(entry):


    return {
        "accession": entry["primaryAccession"],
        "organism": entry["organism"]["scientificName"],
        "kingdom": get_kingdom(entry),
        "length": entry["sequence"]["length"],
    }

### Get signals

Return a valid experimentally supported signal feature, or None.

In [9]:
def get_valid_signal(entry):

    for feature in entry.get("features", []):

        if feature.get("type") != "Signal":
            continue

        if "Not cleaved" in feature.get("description", ""):
            continue

        end_info = feature.get("location", {}).get("end", {})
        end = end_info.get("value")

        if end is None or end_info.get("modifier") == "UNKNOWN":
            continue

        if end < 14:
            continue

        experimental = any(
            evidence.get("evidenceCode") == "ECO:0000269"
            for evidence in feature.get("evidences", [])
        )

        if experimental:
            return feature

    return None

### Check TM

In [10]:
def has_tm_in_first_90(entry):

    return any(
        feature.get("type") == "Transmembrane"
        and feature.get("location", {})
                   .get("start", {})
                   .get("value") is not None
        and feature["location"]["start"]["value"] <= 90

        for feature in entry.get("features", [])
    )

### Positive and negative functions

In [11]:
def positive_record(entry):

    signal = get_valid_signal(entry)

    if signal is None:
        return None

    return {
        **base_record(entry),
        "cleavage_pos": signal["location"]["end"]["value"],
    }


def negative_record(entry):

    if any(
        feature.get("type") == "Signal"
        for feature in entry.get("features", [])
    ):
        return None

    return {
        **base_record(entry),
        "has_tm_in_90": has_tm_in_first_90(entry),
    }

### Save dataset

Fetch, filter and save a UniProt dataset as TSV + FASTA

In [12]:
def save_dataset(query, record_fn, columns, output_prefix):

    total = 0
    kept = 0

    tsv_path = f"{output_prefix}.tsv"
    fasta_path = f"{output_prefix}.fasta"

    with open(
        tsv_path, "w", newline="", encoding="utf-8"
    ) as tsv_out, open(
        fasta_path, "w", encoding="utf-8"
    ) as fasta_out:

        writer = csv.DictWriter(
            tsv_out,
            fieldnames=columns,
            delimiter="\t",
        )
        writer.writeheader()

        for entry in iter_uniprot_entries(query):

            total += 1
            record = record_fn(entry)

            if record is None:
                continue

            kept += 1
            writer.writerow(record)

            accession = entry["primaryAccession"]
            sequence = entry["sequence"]["value"]

            fasta_out.write(
                f">{accession}\n{sequence}\n"
            )

    print(
        f"Done! Total retrieved: {total}, "
        f"passed filter: {kept}"
    )
    print(f"TSV saved to: {tsv_path}")
    print(f"FASTA saved to: {fasta_path}")

## Run!

In [13]:
save_dataset(
    query=positive_query,
    record_fn=positive_record,
    columns=[
        "accession",
        "organism",
        "kingdom",
        "length",
        "cleavage_pos",
    ],
    output_prefix="positive_dataset",
)


save_dataset(
    query=negative_query,
    record_fn=negative_record,
    columns=[
        "accession",
        "organism",
        "kingdom",
        "length",
        "has_tm_in_90",
    ],
    output_prefix="negative_dataset",
)

Done! Total retrieved: 2972, passed filter: 2957
TSV saved to: positive_dataset.tsv
FASTA saved to: positive_dataset.fasta
Done! Total retrieved: 20975, passed filter: 20975
TSV saved to: negative_dataset.tsv
FASTA saved to: negative_dataset.fasta
